# Module 2 · Unsupervised And Manifold Learning

**Track:** AI Engineering Core  
**Objective:** Master the principles and applications of Unsupervised And Manifold Learning.

---



### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** No senior architect feeds raw t-SNE/UMAP results into a downstream model. They destroy global metrics. For data engineering, we use clustering for **Target Leakage detection** and **Outlier Removal**. If your data doesn't naturally cluster, it's often a sign that your features are missing a critical causal signal.

**Mental Model:** 
- **K-Means:** A hard-boundary ruler. Every point belongs to exactly one centroid.
- **GMM:** A soft-colored lantern. Points near the center are 'mostly red', but points in between can be '30% red, 70% blue' (Soft Assignment).
- **DBSCAN:** A density detective. If a point has enough neighbors within $\epsilon$, it belongs. Isolated points are **noise** — no forced assignment.
- **Spectral Clustering:** Uses the graph Laplacian to discover non-convex clusters like concentric rings.

**Common Junior Pitfall:** Not scaling data before clustering. K-Means and GMM are distance-based. If one feature has scale [0, 1] and another [0, 1,000,000], clustering focuses ONLY on the wide feature.

---


## 📑 Table of Contents

  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1. K-Means — Centroid Clustering](#1-k-means--centroid-clustering)
  * [1.1 Elbow Method & Silhouette Analysis](#11-elbow-method--silhouette-analysis)
* [2. Gaussian Mixture Models — Soft Clustering](#2-gaussian-mixture-models--soft-clustering)
* [3. DBSCAN & HDBSCAN — Density-Based Clustering](#3-dbscan--hdbscan--density-based-clustering)
* [4. Spectral Clustering — Graph-Based Discovery](#4-spectral-clustering--graph-based-discovery)
* [5. Anomaly Detection — Isolation Forest](#5-anomaly-detection--isolation-forest)
* [6. UMAP — Topological Manifold Projection](#6-umap--topological-manifold-projection)
* [7. Real-World Pipeline — Customer Segmentation](#7-real-world-pipeline--customer-segmentation)
* [✅ Knowledge Check](#knowledge-check)
* [🔨 Practical Practice](#practical-practice)


## 1. K-Means — Centroid Clustering

K-Means minimizes **Inertia** (sum of squared distances to centroids). It is fast but only works for spherical clusters.

**Algorithm:**
1. Initialize $k$ centroids (K-Means++ for smart initialization)
2. Assign each point to nearest centroid
3. Recompute centroids as cluster means
4. Repeat until convergence


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_circles, make_moons, load_digits
from sklearn.cluster import KMeans, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.stats import multivariate_normal

np.random.seed(42)
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, cluster_std=0.6, random_state=42)

# K-Means from scratch
def kmeans_scratch(X, k=3, max_iter=20):
    # K-Means++ initialization
    centroids = [X[np.random.randint(len(X))]]
    for _ in range(k-1):
        dists = np.min([np.sum((X - c)**2, axis=1) for c in centroids], axis=0)
        probs = dists / dists.sum()
        centroids.append(X[np.random.choice(len(X), p=probs)])
    centroids = np.array(centroids)
    
    for _ in range(max_iter):
        labels = np.argmin(np.linalg.norm(X[:, None] - centroids, axis=2), axis=1)
        new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(k)])
        if np.allclose(centroids, new_centroids): break
        centroids = new_centroids
    return labels, centroids

labels, centroids = kmeans_scratch(X_blobs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap='Set2', s=30)
axes[0].scatter(centroids[:, 0], centroids[:, 1], marker='X', s=200, c='red', edgecolors='black')
axes[0].set_title('K-Means++ (From Scratch)', fontsize=13)

# Show K-Means failure case: elongated clusters
rng = np.random.RandomState(42)
X_elongated = np.vstack([
    rng.randn(100, 2) @ [[3, 0], [0, 0.3]] + [0, 0],
    rng.randn(100, 2) @ [[0.3, 0], [0, 3]] + [5, 5]
])
km_fail = KMeans(n_clusters=2, random_state=42).fit_predict(X_elongated)
axes[1].scatter(X_elongated[:, 0], X_elongated[:, 1], c=km_fail, cmap='Set2', s=30)
axes[1].set_title('K-Means FAILS: Elongated Clusters', fontsize=13)
plt.tight_layout()
plt.show()

"""What just happened?
Left: K-Means++ perfectly clusters spherical blobs. Right: It FAILS on elongated
clusters because K-Means assumes isotropic (equal-spread) distances. GMM handles
this by learning per-cluster covariance matrices.
"""

### 1.1 Elbow Method & Silhouette Analysis

**Elbow Method:** Plot inertia vs K. The 'elbow' is where adding more clusters gives diminishing returns.

**Silhouette Score:** Measures how similar a point is to its own cluster vs the nearest neighbor cluster. Range [-1, 1]: high = well-clustered, 0 = on boundary, negative = wrong cluster.


In [ ]:
k_range = range(2, 10)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(X_blobs)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_blobs, labels_k))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_range), inertias, 'o-', color='#e74c3c', linewidth=2, markersize=8)
axes[0].set_xlabel('K', fontsize=12)
axes[0].set_ylabel('Inertia', fontsize=12)
axes[0].set_title('Elbow Method', fontsize=14)
axes[0].axvline(3, color='gray', linestyle='--', alpha=0.5)

axes[1].plot(list(k_range), silhouettes, 'o-', color='#3498db', linewidth=2, markersize=8)
axes[1].set_xlabel('K', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score', fontsize=14)
axes[1].axvline(3, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Optimal K Selection: Both Methods Point to K=3', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

"""What just happened?
Both methods agree: K=3 is optimal. The elbow shows diminishing returns after K=3.
The silhouette score peaks at K=3. When they disagree, trust the silhouette — it
directly measures cluster quality rather than just within-cluster spread.
"""

## 2. Gaussian Mixture Models — Soft Clustering

GMMs model each cluster as a Gaussian $N(\mu_k, \Sigma_k)$, optimized via **Expectation-Maximization (EM)**:
1. **E-Step:** Calculate responsibility $r_{ik}$ — probability of point $i$ belonging to cluster $k$
2. **M-Step:** Update means, covariances, and mixing weights using weighted statistics

🎓 **Socratic Deep Check:**
> *"If you set every GMM covariance to the Identity matrix and force hard (0/1) responsibilities, why does GMM become exactly K-Means?"*


In [ ]:
from sklearn.mixture import GaussianMixture

# GMM handles elongated clusters where K-Means fails
gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=42)
gmm_labels = gmm.fit_predict(X_elongated)
gmm_probs = gmm.predict_proba(X_elongated)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# K-Means failure
axes[0].scatter(X_elongated[:, 0], X_elongated[:, 1], c=km_fail, cmap='Set2', s=30)
axes[0].set_title('K-Means: Wrong!', fontsize=13)

# GMM hard assignment
axes[1].scatter(X_elongated[:, 0], X_elongated[:, 1], c=gmm_labels, cmap='Set2', s=30)
axes[1].set_title('GMM: Correct!', fontsize=13)

# GMM soft assignment (uncertainty)
uncertainty = 1 - gmm_probs.max(axis=1)
sc = axes[2].scatter(X_elongated[:, 0], X_elongated[:, 1], c=uncertainty, cmap='YlOrRd', s=30)
plt.colorbar(sc, ax=axes[2], label='Uncertainty')
axes[2].set_title('GMM Uncertainty: Borderline Points', fontsize=13)

plt.suptitle('GMM vs K-Means on Elongated Clusters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# BIC for model selection
bics = [GaussianMixture(n_components=k, random_state=42).fit(X_blobs).bic(X_blobs)
        for k in range(1, 8)]
print(f"BIC-optimal K: {np.argmin(bics)+1} (BIC penalizes complexity like AIC)")

"""What just happened?
K-Means fails on elongated clusters because it assumes spherical shape. GMM learns
per-cluster covariance, correctly handling elliptical clusters. The uncertainty plot
(right) shows which points the model is unsure about — critical for flagging
ambiguous cases in production.
"""

## 3. DBSCAN & HDBSCAN — Density-Based Clustering

**DBSCAN** discovers clusters of arbitrary shape by looking at local density:
- **Core point:** Has $\ge$ `min_samples` neighbors within `eps`
- **Border point:** Within `eps` of a core point but not itself core
- **Noise point:** Neither — classified as outlier (label=-1)

**HDBSCAN** removes the `eps` parameter entirely, automatically detecting clusters at varying densities.

📈 **Production Signal:** DBSCAN is widely used for geospatial clustering (finding store clusters in GPS data) and anomaly detection (noise points = anomalies).


In [ ]:
try:
    import hdbscan
except ImportError:
    print("Installing hdbscan...")
    !pip install -q hdbscan
    import hdbscan

# Create data with varying density + noise
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)
noise = np.random.uniform(-1.5, 2.5, (30, 2))  # Add noise points
X_noisy = np.vstack([X_moons, noise])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# K-Means (fails on moons)
km_moons = KMeans(n_clusters=2, random_state=42).fit_predict(X_noisy)
axes[0].scatter(X_noisy[:, 0], X_noisy[:, 1], c=km_moons, cmap='Set2', s=20)
axes[0].set_title('K-Means: Fails on Moons + Noise', fontsize=13)

# DBSCAN
db = DBSCAN(eps=0.2, min_samples=5)
db_labels = db.fit_predict(X_noisy)
n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = (db_labels == -1).sum()
axes[1].scatter(X_noisy[:, 0], X_noisy[:, 1], c=db_labels, cmap='Set2', s=20)
axes[1].scatter(X_noisy[db_labels==-1, 0], X_noisy[db_labels==-1, 1], 
                c='red', marker='x', s=50, label=f'Noise ({n_noise})')
axes[1].set_title(f'DBSCAN: {n_clusters} Clusters + Noise Detection', fontsize=13)
axes[1].legend()

# HDBSCAN
hdb = hdbscan.HDBSCAN(min_cluster_size=15)
hdb_labels = hdb.fit_predict(X_noisy)
n_hdb = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
axes[2].scatter(X_noisy[:, 0], X_noisy[:, 1], c=hdb_labels, cmap='Set2', s=20)
axes[2].scatter(X_noisy[hdb_labels==-1, 0], X_noisy[hdb_labels==-1, 1],
                c='red', marker='x', s=50, label=f'Noise ({(hdb_labels==-1).sum()})')
axes[2].set_title(f'HDBSCAN: {n_hdb} Clusters (No eps needed!)', fontsize=13)
axes[2].legend()

plt.suptitle('Density-Based Clustering: Handling Non-Convex Shapes + Noise', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

"""What just happened?
K-Means splits moons horizontally (wrong). DBSCAN correctly follows the density of
each moon and labels isolated points as noise. HDBSCAN does the same without needing
the eps parameter — it automatically adapts to varying density.
"""

## 4. Spectral Clustering — Graph-Based Discovery

Uses the **Graph Laplacian** to find clusters in data that is not separable in the original space:
1. Build similarity graph $W_{ij} = e^{-||x_i-x_j||^2 / 2\sigma^2}$
2. Compute Laplacian $L = D - W$
3. Use the smallest eigenvectors of $L$ as new coordinates
4. Apply K-Means in this spectral space


In [ ]:
X_circles, y_circles = make_circles(n_samples=300, factor=0.4, noise=0.05, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# K-Means fails
km_circ = KMeans(n_clusters=2, random_state=42).fit_predict(X_circles)
axes[0].scatter(X_circles[:, 0], X_circles[:, 1], c=km_circ, cmap='Set2', s=30)
axes[0].set_title('K-Means: FAILS on Rings', fontsize=13)

# Spectral Clustering
sc = SpectralClustering(n_clusters=2, affinity='rbf', gamma=10, random_state=42)
sc_labels = sc.fit_predict(X_circles)
axes[1].scatter(X_circles[:, 0], X_circles[:, 1], c=sc_labels, cmap='Set2', s=30)
axes[1].set_title('Spectral Clustering: Correct!', fontsize=13)

# Show the spectral embedding
from scipy.linalg import eigh
from sklearn.metrics.pairwise import rbf_kernel
W = rbf_kernel(X_circles, gamma=10)
D = np.diag(W.sum(axis=1))
L = D - W
vals, vecs = eigh(L, subset_by_index=[0, 1])
axes[2].scatter(vecs[:, 0], vecs[:, 1], c=y_circles, cmap='Set2', s=30)
axes[2].set_title('Spectral Embedding: Rings Become Blobs!', fontsize=13)

plt.suptitle('Spectral Clustering: Graph Theory Saves the Day', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

"""What just happened?
K-Means fails because rings share the same centroid. The spectral embedding (right)
transforms the data using the graph Laplacian eigenvectors, mapping the concentric
rings into two linearly separable blobs. K-Means then works perfectly in this space.
"""

## 5. Anomaly Detection — Isolation Forest

**Isolation Forest** detects anomalies by how quickly they can be 'isolated' by random splits. Anomalies sit in sparse regions and require fewer splits.

📈 **Production Signal:** Used for fraud detection, network intrusion, and manufacturing defect detection.


In [ ]:
from sklearn.ensemble import IsolationForest

# Normal data + injected anomalies
X_normal = np.random.randn(300, 2) * 0.5
X_anomalies = np.random.uniform(-3, 3, (15, 2))  # Sparse outliers
X_all = np.vstack([X_normal, X_anomalies])
y_true_anom = np.array([1]*300 + [-1]*15)  # 1=normal, -1=anomaly

iso_forest = IsolationForest(contamination=0.05, random_state=42)
y_pred_anom = iso_forest.fit_predict(X_all)
scores = iso_forest.decision_function(X_all)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Decision boundary
xx, yy = np.meshgrid(np.linspace(-4, 4, 200), np.linspace(-4, 4, 200))
Z = iso_forest.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[0].contourf(xx, yy, Z, levels=20, cmap='RdYlGn', alpha=0.3)
axes[0].scatter(X_all[y_pred_anom==1, 0], X_all[y_pred_anom==1, 1], c='#2ecc71', s=20, label='Normal')
axes[0].scatter(X_all[y_pred_anom==-1, 0], X_all[y_pred_anom==-1, 1], c='#e74c3c', s=80, 
                marker='X', edgecolors='black', label='Anomaly')
axes[0].set_title('Isolation Forest: Anomaly Detection', fontsize=13)
axes[0].legend()

# Anomaly score distribution
axes[1].hist(scores[y_true_anom==1], bins=30, alpha=0.7, color='#2ecc71', label='Normal', density=True)
axes[1].hist(scores[y_true_anom==-1], bins=15, alpha=0.7, color='#e74c3c', label='True Anomaly', density=True)
axes[1].set_title('Anomaly Score Distribution', fontsize=13)
axes[1].set_xlabel('Anomaly Score (lower = more anomalous)')
axes[1].legend()

plt.tight_layout()
plt.show()

"""What just happened?
Isolation Forest detected outliers by measuring how easily each point can be
separated from the rest. The contour plot shows the anomaly score surface — points
far from dense regions get low scores (red) and are flagged as anomalies.
"""

## 6. UMAP — Topological Manifold Projection

**UMAP** builds a fuzzy topological graph of high-D data and optimizes a low-D layout preserving both local AND global structure. Advantages over t-SNE:
1. **Speed:** 10x faster using SGD
2. **Structure:** Preserves global distances better
3. **Inference:** Can transform new, unseen points via `.transform()`


In [ ]:
import time
from sklearn.manifold import TSNE

try:
    from umap import UMAP
except ImportError:
    print("Installing umap-learn...")
    !pip install -q umap-learn
    from umap import UMAP

digits = load_digits()
X_dig = digits.data[:1500]
y_dig = digits.target[:1500]

start = time.time()
X_tsne = TSNE(random_state=42, perplexity=30).fit_transform(X_dig)
time_tsne = time.time() - start

start = time.time()
X_umap = UMAP(random_state=42, n_neighbors=15).fit_transform(X_dig)
time_umap = time.time() - start

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, X_proj, name, t in [
    (axes[0], X_tsne, 't-SNE', time_tsne),
    (axes[1], X_umap, 'UMAP', time_umap)
]:
    scatter = ax.scatter(X_proj[:, 0], X_proj[:, 1], c=y_dig, s=10, cmap='Spectral')
    ax.set_title(f'{name} ({t:.1f}s)', fontsize=14)
    ax.set_xticks([]); ax.set_yticks([])
plt.colorbar(scatter, ax=axes, label='Digit Class')
plt.suptitle('Manifold Visualization: 64D Digits → 2D', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"t-SNE: {time_tsne:.1f}s | UMAP: {time_umap:.1f}s | Speedup: {time_tsne/time_umap:.1f}x")

"""What just happened?
Both methods reduce 64D digit data to 2D. UMAP is faster and preserves more global
structure — similar digits (1 and 7, 3 and 8) cluster near each other. t-SNE creates
isolated bubbles with arbitrary placement. For exploration, UMAP is now the standard.
"""

## 7. Real-World Pipeline — Customer Segmentation

A complete unsupervised ML pipeline:
1. **Scale** features
2. **Select K** via silhouette analysis
3. **Cluster** with the best algorithm
4. **Profile** clusters for business insight


In [ ]:
from sklearn.datasets import make_blobs

# Simulate customer data: [Recency, Frequency, Monetary]
np.random.seed(42)
rfm_data = np.vstack([
    np.random.normal([10, 50, 500], [3, 10, 100], (200, 3)),  # High-value
    np.random.normal([60, 5, 50], [15, 3, 30], (300, 3)),      # Dormant
    np.random.normal([30, 20, 200], [8, 5, 50], (250, 3)),     # Mid-tier
])

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_data)

# Optimal K via silhouette
sil_scores = {k: silhouette_score(rfm_scaled, KMeans(n_clusters=k, random_state=42).fit_predict(rfm_scaled))
              for k in range(2, 8)}
best_k = max(sil_scores, key=sil_scores.get)

km_rfm = KMeans(n_clusters=best_k, random_state=42)
rfm_labels = km_rfm.fit_predict(rfm_scaled)

# Profile clusters
print(f"Optimal K: {best_k}\n")
print(f"{'Cluster':<10} {'Recency':>10} {'Frequency':>12} {'Monetary':>12} {'Size':>8}")
print('-' * 55)
for c in range(best_k):
    mask = rfm_labels == c
    print(f"{'Cluster '+str(c):<10} {rfm_data[mask, 0].mean():>10.1f} "
          f"{rfm_data[mask, 1].mean():>12.1f} {rfm_data[mask, 2].mean():>12.1f} {mask.sum():>8}")

"""What just happened?
We built a complete customer segmentation pipeline: scale → select K → cluster →
profile. The three segments emerge naturally: High-value (recent, frequent, high
spend), Dormant (old, rare, low spend), and Mid-tier. In production, each segment
gets a tailored marketing strategy.
"""

---
## ✅ Knowledge Check

### Q1: GMM vs K-Means
<details><summary>👉 Answer</summary>
K-Means is a special case of GMM where covariances are identity matrices and assignments are hard (0/1). GMM generalizes this with full covariance and soft probabilities, handling elongated and overlapping clusters.
</details>

### Q2: When does DBSCAN beat GMM?
<details><summary>👉 Answer</summary>
DBSCAN excels when clusters have arbitrary shapes (moons, spirals) and data contains noise/outliers. Unlike GMM, it doesn't assume Gaussian shapes and automatically identifies noise points. However, it struggles with clusters of varying density.
</details>

### Q3: Why not use t-SNE embeddings as features?
<details><summary>👉 Answer</summary>
t-SNE destroys global distances — points far apart in high-D may end up close in 2D. The embedding is not metric-preserving, so using it as features for a downstream model introduces systematic distortion. UMAP is slightly better but still lossy.
</details>


---
## 🔨 Practical Practice

### Tier 1: Beginner
1. Implement **K-Means++** initialization and compare final inertia against random initialization over 10 trials.
2. Run DBSCAN with `eps` values [0.1, 0.3, 0.5, 1.0] on the moons dataset. Observe how the number of clusters and noise points change.

### Tier 2: Intermediate
1. **Silhouette Plot:** Build a silhouette plot (per-sample silhouette bars) for K=2,3,4 on a blob dataset. Identify which K produces the most uniform silhouette widths.
2. **UMAP Out-of-Sample:** Fit UMAP on digits [0-7], then use `.transform()` to project digits [8,9]. Do they land near visually similar digits?

### Tier 3: Advanced
1. **GMM Full vs Diagonal:** Implement GMM with full vs diagonal covariance. Prove that diagonal forces axis-aligned clusters. When does this help vs hurt?
2. **Hierarchical Density Estimation:** Implement the HDBSCAN condensed tree manually. Show how it selects clusters at the optimal density level.
